In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve
)
from ultralytics import YOLO
from collections import Counter

model = YOLO('/Users/ahmadfauzi/Downloads/Detection.pt')
metrics = model.val(
    data='/Users/ahmadfauzi/Downloads/Combination/data.yaml',
    project='/Users/ahmadfauzi/Downloads/yolo_results',
    name='test_fix_combination_test',
    save=True,
    save_txt=True
)

pred_dir = '/Users/ahmadfauzi/Downloads/yolo_results/test_fix_combination/labels'
true_dir = '/Users/ahmadfauzi/Downloads/Combination/labels/val'

class_names = [
    'Eritroid', 'Lymphoid', 'Megakaryoid',
    'Myeloid', 'Plasmacytoid', 'Mitosis', 'Artefact'
]
n_classes = len(class_names)
labels = list(range(n_classes))

def filter_classes(y_true, y_pred, y_true_bin, y_pred_bin, min_samples=2):
    class_counts = Counter(y_true)
    keep_classes = [cls for cls, count in class_counts.items() if count >= min_samples]
    keep_classes = sorted(keep_classes)
    print(f"Keeping classes with ≥{min_samples} samples:", keep_classes)
    class_mapping = {old: new for new, old in enumerate(keep_classes)}
    reverse_map = {v: k for k, v in class_mapping.items()}
    y_true_filtered = []
    y_pred_filtered = []
    y_true_bin_filtered = []
    y_pred_bin_filtered = []
    for i, t in enumerate(y_true):
        if t in keep_classes:
            new_idx = class_mapping[t]
            p = y_pred[i]
            if p in keep_classes:
                y_true_filtered.append(new_idx)
                y_pred_filtered.append(class_mapping[p])
                y_true_bin_filtered.append(y_true_bin[i])
                y_pred_bin_filtered.append(y_pred_bin[i])
    return (
        np.array(y_true_filtered),
        np.array(y_pred_filtered),
        np.array(y_true_bin_filtered),
        np.array(y_pred_bin_filtered),
        keep_classes,
        [class_names[k] for k in keep_classes]
    )

y_true, y_pred = [], []
y_true_bin, y_pred_bin = [], []

for fname in os.listdir(true_dir):
    if not fname.endswith('.txt'):
        continue
    true_path = os.path.join(true_dir, fname)
    pred_path = os.path.join(pred_dir, fname)
    with open(true_path, 'r') as f:
        true_classes = [int(line.strip().split()[0]) for line in f]
    pred_classes = []
    if os.path.exists(pred_path):
        with open(pred_path, 'r') as f:
            pred_classes = [int(line.strip().split()[0]) for line in f]
    for t in true_classes:
        predicted = pred_classes[0] if pred_classes else -1
        if predicted != -1:
            y_true.append(t)
            y_pred.append(predicted)
            t_bin = [0] * n_classes
            p_bin = [0] * n_classes
            t_bin[t] = 1
            p_bin[predicted] = 1
            y_true_bin.append(t_bin)
            y_pred_bin.append(p_bin)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_true_bin = np.array(y_true_bin)
y_pred_bin = np.array(y_pred_bin)
print("\n📊 Ground Truth and Prediction Counts (Before Filtering):")

# Ground truth counts per class
gt_counts = Counter(y_true)
print("Ground Truth (y_true):")
for i in range(n_classes):
    print(f"  {i}: {class_names[i]:<15} → {gt_counts.get(i, 0)}")

# Predicted counts per class
pred_counts = Counter(y_pred)
print("\nPredictions (y_pred):")
for i in range(n_classes):
    print(f"  {i}: {class_names[i]:<15} → {pred_counts.get(i, 0)}")

# Check 1-hot true presence (used by AUC)
print("\n🧪 AUC-relevant One-Hot Presence (y_true_bin):")
for i in range(n_classes):
    positives = int(np.sum(y_true_bin[:, i]))
    print(f"  {i}: {class_names[i]:<15} → {positives} positives")


y_true_filt, y_pred_filt, y_true_bin_filt, y_pred_bin_filt, keep_ids, keep_names = filter_classes(
    y_true, y_pred, y_true_bin, y_pred_bin, min_samples=2
)
y_true = y_true_filt
y_pred = y_pred_filt
y_true_bin = np.array(y_true_bin_filt)
y_pred_bin = np.array(y_pred_bin_filt)
class_names = keep_names
n_classes = len(class_names)
labels = list(range(n_classes))

print("\n✅ Evaluation Sample Summary:")
print(f"Number of ground truth samples used (y_true): {len(y_true)}")
print(f"Number of predicted samples used (y_pred): {len(y_pred)}")

cm = confusion_matrix(y_true, y_pred, labels=labels)
sensitivity = recall_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
precision = precision_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
f1 = f1_score(y_true, y_pred, labels=labels, average=None, zero_division=0)

specificity = []
for i in labels:
    tn = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
    fp = cm[:, i].sum() - cm[i, i]
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    specificity.append(spec)

df = pd.DataFrame({
    'Class Index': labels,
    'Class Name': class_names,
    'Precision': np.round(precision, 3),
    'Sensitivity (Recall)': np.round(sensitivity, 3),
    'Specificity': np.round(specificity, 3),
    'F1 Score': np.round(f1, 3)
})

excel_path = '/Users/ahmadfauzi/Downloads/metrics_report_combination.xlsx'
df.to_excel(excel_path, index=False)
print(f"Metrics saved to Excel: {excel_path}")

fpr, tpr, roc_auc = {}, {}, {}
for i in labels:
    try:
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_pred_bin[:, i])
        roc_auc[i] = roc_auc_score(y_true_bin[:, i], y_pred_bin[:, i])
    except ValueError:
        fpr[i], tpr[i], roc_auc[i] = [0], [0], 0.0

all_fpr = np.unique(np.concatenate([fpr[i] for i in labels]))
mean_tpr = np.zeros_like(all_fpr)
for i in labels:
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= n_classes
macro_auc = np.mean(list(roc_auc.values()))

plt.figure(figsize=(8, 6))
for i in labels:
    plt.plot(fpr[i], tpr[i], label=f"{class_names[i]} (AUC = {roc_auc[i]:.2f})")

plt.plot(all_fpr, mean_tpr, linestyle='--', label=f'Macro-Average (AUC = {macro_auc:.2f})', color='black')
plt.plot([0, 1], [0, 1], linestyle=':', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-Class ROC-AUC Curve')
plt.legend(loc='lower right')
plt.grid(True)

roc_path = '/Users/ahmadfauzi/Downloads/roc_auc_curve_combination.png'
plt.savefig(roc_path, dpi=300)
print(f"AUC curve saved to: {roc_path}")